In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [2]:
from huggingface_hub import login
from transformers import AutoModel
from PIL import Image
import torch
from torchvision import transforms
from typing import Union
import os


class ConchFeatureExtractorCrops:
    def __init__(self, token: str = ""):
        # Login to Hugging Face (for gated models)
        login(token=hf_token)

        # Load TITAN / CONCH
        self.model = AutoModel.from_pretrained(
            "MahmoodLab/TITAN",
            trust_remote_code=True
        )

        self.conch, self.eval_transform = self.model.return_conch()

        # FiveCrop transform
        self.fivecrop_transform = transforms.Compose([            
            transforms.FiveCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self.eval_transform(c) for c in crops])
            )
        ])

        # TenCrop transform
        self.tencrop_transform = transforms.Compose([
            transforms.TenCrop(256),
            transforms.Lambda(
                lambda crops: torch.stack([self.eval_transform(c) for c in crops])
            )
        ])

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.conch.to(self.device)
        self.conch.eval()

    def _load_image(self, image: Union[str, Image.Image]) -> Image.Image:
        if isinstance(image, str):
            if not os.path.exists(image):
                raise FileNotFoundError(f"Image not found: {image}")
            image = Image.open(image).convert("RGB")
        elif isinstance(image, Image.Image):
            image = image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image type: {type(image)}")
        return image

    def extract_fivecrop_features(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Extract features from 5 crops.
        Returns: (5, D)
        """
        image = self._load_image(image)

        crops_tensor = self.fivecrop_transform(image).to(self.device)  # (5, C, H, W)

        with torch.inference_mode():
            features = self.conch(crops_tensor)  # (5, D)

        return features.cpu()

    def extract_tencrop_features(self, image: Union[str, Image.Image]) -> torch.Tensor:
        """
        Extract features from 10 crops.
        Returns: (10, D)
        """
        image = self._load_image(image)

        crops_tensor = self.tencrop_transform(image).to(self.device)  # (10, C, H, W)

        with torch.inference_mode():
            features = self.conch(crops_tensor)  # (10, D)

        return features.cpu()


<h1>Testing<h1>

In [3]:
# CONCH FiveCrop / TenCrop feature extraction script
import glob
import os
from tqdm import tqdm
import torch

# =========================
# Set paths
# =========================
patch_root = r"D:\Aamir Gulzar\KSA_project2\dataset\test_patch_data"
save_root = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\CONCH_tencrop_test"

# =========================
# Initialize CONCH extractor
# =========================
extractor = ConchFeatureExtractorCrops()

# =========================
# Collect all PNG patches
# =========================
all_patch_paths = glob.glob(os.path.join(patch_root, "*", "*.png"))

print(f"Found {len(all_patch_paths)} patches")

# =========================
# Loop through patches
# =========================
for patch_path in tqdm(all_patch_paths):
    try:
        # ---- Choose ONE ----

        # Five-crop feature extraction
        # features = extractor.extract_fivecrop_features(patch_path)  # (5, D)

        # Ten-crop feature extraction
        features = extractor.extract_tencrop_features(patch_path)     # (10, D)

        # =========================
        # Preserve folder structure
        # =========================
        relative_path = os.path.relpath(patch_path, patch_root)
        slide_folder, patch_filename = os.path.split(relative_path)
        patch_base = os.path.splitext(patch_filename)[0]

        save_dir = os.path.join(save_root, slide_folder)
        os.makedirs(save_dir, exist_ok=True)

        save_path = os.path.join(save_dir, patch_base + ".pt")

        # =========================
        # Save features
        # =========================
        torch.save(features, save_path)

    except Exception as e:
        print(f"❌ Error processing {patch_path}: {e}")


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
Found 6164 patches


  0%|          | 0/6164 [00:00<?, ?it/s]C:\Users\datainsight\.cache\huggingface\modules\transformers_modules\MahmoodLab\TITAN\dac6773d9961cfc75503440676ff157a2c6e8d2e\conch_v1_5.py:144: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(
100%|██████████| 6164/6164 [20:12<00:00,  5.08it/s]


In [3]:
import os
import glob
import pandas as pd
import torch
from tqdm import tqdm


def run_full_pipeline_CONCH_crops(
    patch_data_dir: str,
    extractor,  # Instance of ConchFeatureExtractorCrops
    non_white_csv: str,
    output_root: str,
    use_tencrop: bool = True
):
    """
    Full CONCH feature extraction pipeline with resume support.
    Extracts 5-crop or 10-crop features for non-white patches only.
    """

    # =========================
    # Load non-white metadata
    # =========================
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df["patch_name"].astype(str))

    # =========================
    # Loop over slide directories
    # =========================
    slide_dirs = glob.glob(os.path.join(patch_data_dir, "*"))

    for slide_dir in tqdm(slide_dirs, desc="🧩 Processing slides", unit="slide", position=0):
        slide_name = os.path.basename(slide_dir)

        # Output directory for this slide
        patch_feat_dir = os.path.join(output_root, slide_name)
        os.makedirs(patch_feat_dir, exist_ok=True)

        # =========================
        # Collect valid non-white patches
        # =========================
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        valid_patch_paths = [
            p for p in all_patches
            if os.path.basename(p) in valid_patch_names
        ]

        if not valid_patch_paths:
            tqdm.write(f"  ❌ No valid non-white patches found for {slide_name}")
            continue

        # =========================
        # Resume logic (skip processed patches)
        # =========================
        existing_feats = {
            os.path.splitext(f.name)[0]
            for f in os.scandir(patch_feat_dir)
            if f.name.endswith(".pt")
        }

        to_process = [
            p for p in valid_patch_paths
            if os.path.splitext(os.path.basename(p))[0] not in existing_feats
        ]

        if not to_process:
            tqdm.write(
                f"  ⏭️ Skipped: All {len(valid_patch_paths)} patches already processed for {slide_name}"
            )
            continue

        # =========================
        # Patch-level processing
        # =========================
        for patch_path in tqdm(
            to_process,
            desc=f"🔄 {slide_name}",
            unit="patch",
            position=1,
            leave=False
        ):
            patch_name = os.path.splitext(os.path.basename(patch_path))[0]
            save_path = os.path.join(patch_feat_dir, patch_name + ".pt")

            try:
                if use_tencrop:
                    features = extractor.extract_tencrop_features(patch_path)  # (10, D)
                else:
                    features = extractor.extract_fivecrop_features(patch_path)  # (5, D)

                torch.save(features, save_path)

            except Exception as e:
                tqdm.write(f"  ❌ Failed to process patch {patch_path}: {e}")
                continue

        tqdm.write(
            f"  ✅ Done: Processed {len(to_process)} new patches "
            f"(total valid: {len(valid_patch_paths)}) for {slide_name}"
        )


In [ ]:
extractor = ConchFeatureExtractorCrops()

run_full_pipeline_CONCH_crops(
    patch_data_dir=r"D:\Aamir Gulzar\KSA_project2\dataset\patch_data",
    extractor=extractor,
    non_white_csv=r"D:\Aamir Gulzar\KSA_project2\dataset\final_merged_without_white.csv",
    output_root=r"D:\Aamir Gulzar\KSA_project2\dataset\Features\CONCH_tencrop"
)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
